# Stage 3: Causal Disentanglement

**Objective:** Establish *causal* (not merely correlational) evidence that gender-correlated shortcut features mediate the path from gender to model predictions. Move beyond "the model encodes shortcuts" (Stage 2) to "the model *causally relies* on shortcuts, and this reliance is gender-biased."

**Methods:**
1. **Structural Causal Model (SCM):** Formalize the DAG linking gender, shortcuts, clinical signal, embeddings, and predictions
2. **Causal Mediation Analysis:** Estimate what fraction of the gender→prediction association flows through each shortcut (ACME, ADE, proportion mediated)
3. **Concept Erasure (INLP):** Remove shortcut information from [CLS] embeddings via Iterative Nullspace Projection, then re-probe for gender and label
4. **Counterfactual Interventions:** Apply a fixed label classifier to shortcut-erased embeddings; measure prediction flip rates stratified by gender
5. **Conditional Independence Tests:** Test whether gender provides information about labels beyond what shortcuts already capture

**Inputs:** `data/features_14_extracted.pkl`, frozen DistilRoBERTa [CLS] embeddings
**Shortcut candidates (from Stage 1):** `negative_emotion`, `certainty`, `question_density`

In [ ]:
import os, warnings, time
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.special import expit
from scipy.stats import chi2
from transformers import AutoTokenizer, AutoModel

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 11})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPUS = torch.cuda.device_count() if DEVICE == 'cuda' else 0
print(f'Device: {DEVICE}  |  GPUs: {N_GPUS}')

SHORTCUT_FEATURES = [
    'fp_singular', 'emotional_feeling', 'excl_intensifiers',
    'negative_emotion', 'post_length', 'hedge_density', 'social_relational',
]
ALL_14_FEATURES = [
    'hedge_density', 'fp_singular', 'fp_plural', 'emotional_feeling',
    'social_relational', 'certainty', 'negative_emotion', 'swear_words',
    'excl_intensifiers', 'question_density', 'post_length', 'apology_selfblame',
    'anger', 'body_health',
]

SEED = 42
N_BOOT = 1000
np.random.seed(SEED)
print(f'Shortcut features ({len(SHORTCUT_FEATURES)}): {SHORTCUT_FEATURES}')
print(f'Bootstrap iterations: {N_BOOT}')


In [ ]:
# -- Load Stage 1/2 data ---------------------------------------------------
df = pd.read_pickle('data/stage1/features_14_extracted.pkl')
df['text'] = df['text'].fillna('')
GENDER_COL = 'gender'
df['gender_bin'] = (df[GENDER_COL] == 'female').astype(int)

print(f'Total rows: {len(df):,}')
print(f'Gender: {df[GENDER_COL].value_counts().to_dict()}')
print(f'Labels: {df["binary_label"].value_counts().to_dict()}')

# -- [CLS] embeddings (cached from Stage 2) --------------------------------
CLS_CACHE = 'data/stage2/cls_embeddings.npy'
X_cls_all = np.load(CLS_CACHE)
print(f'\nLoaded cached [CLS]: {X_cls_all.shape}')
assert len(X_cls_all) == len(df), f'Shape mismatch: {len(X_cls_all)} vs {len(df)}'
print('Shape check passed.')


## Part A — Structural Causal Model

We formalize the causal relationships as a Directed Acyclic Graph (DAG). The key paths are:

- **Shortcut pathway (bias):** Gender → Writing-style shortcuts → Embedding → Prediction
- **Clinical pathway (legitimate):** Gender → Clinical condition → Text content → Embedding → Prediction
- **Direct path (tested):** Gender → Prediction (if the model uses demographic markers directly)

The causal question: *How much of the total Gender → Prediction effect is mediated through shortcuts?*

In [ ]:
# -- DAG Visualization -----------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 5.5))
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0.05, 0.95)
ax.axis('off')

pos = {
    'Gender\n(G)':           (0.06, 0.50),
    'Shortcuts\n(S)':        (0.34, 0.78),
    'Clinical\nSignal (C)':  (0.34, 0.22),
    'Embedding\n(E)':        (0.62, 0.50),
    'Prediction\n(\u0176)':  (0.90, 0.50),
}
node_colors = {
    'Gender\n(G)':           '#FF9999',
    'Shortcuts\n(S)':        '#FFD700',
    'Clinical\nSignal (C)':  '#90EE90',
    'Embedding\n(E)':        '#87CEEB',
    'Prediction\n(\u0176)':  '#DDA0DD',
}

for name, (x, y) in pos.items():
    bbox = dict(boxstyle='round,pad=0.4', facecolor=node_colors[name],
                edgecolor='black', lw=2, alpha=0.85)
    ax.text(x, y, name, ha='center', va='center', fontsize=13,
            fontweight='bold', bbox=bbox, zorder=5)

def draw_edge(src, dst, color, ls='-', lw=2.5, label='', y_off=0.05):
    x1, y1 = pos[src]; x2, y2 = pos[dst]
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw,
                                linestyle=ls, shrinkA=30, shrinkB=30))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2 + y_off
        ax.text(mx, my, label, ha='center', va='bottom', fontsize=9,
                color=color, fontstyle='italic')

draw_edge('Gender\n(G)', 'Shortcuts\n(S)', '#CC0000', label='style bias')
draw_edge('Shortcuts\n(S)', 'Embedding\n(E)', '#CC0000', label='encoded')
draw_edge('Gender\n(G)', 'Clinical\nSignal (C)', '#006600', '--', label='base rates')
draw_edge('Clinical\nSignal (C)', 'Embedding\n(E)', '#006600', label='encoded')
draw_edge('Embedding\n(E)', 'Prediction\n(\u0176)', '#333333')

ax.annotate('', xy=(pos['Prediction\n(\u0176)'][0], pos['Prediction\n(\u0176)'][1] - 0.12),
            xytext=(pos['Gender\n(G)'][0], pos['Gender\n(G)'][1] - 0.12),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5,
                            linestyle=':', shrinkA=30, shrinkB=30))
ax.text(0.48, 0.27, 'direct bias?', ha='center', fontsize=9,
        color='gray', fontstyle='italic')

legend = [
    mpatches.Patch(facecolor='#FFD700', edgecolor='black', label='Shortcut pathway (bias)'),
    mpatches.Patch(facecolor='#90EE90', edgecolor='black', label='Clinical pathway (legitimate)'),
    plt.Line2D([0], [0], color='gray', ls=':', lw=1.5, label='Direct bias (tested)'),
]
ax.legend(handles=legend, loc='lower right', fontsize=10, framealpha=0.9)
ax.set_title('Structural Causal Model: Gender -> Shortcuts -> Prediction',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('data/stage3/causal_dag.png', dpi=150, bbox_inches='tight')
print('Saved -> data/stage3/causal_dag.png')
plt.show()


## Part B — Causal Mediation Analysis

Using the **counterfactual mediation framework** (Imai et al., 2010):
- **Treatment:** Gender ($G \in \{0{=}\text{male}, 1{=}\text{female}\}$)
- **Mediator:** Each shortcut feature ($M$, continuous, standardized)
- **Outcome:** Binary mental-health label ($Y$)

Quantities estimated:
- **ACME** (Average Causal Mediation Effect): indirect effect $G \to M \to Y$
- **ADE** (Average Direct Effect): $G \to Y$ not through $M$
- **Proportion mediated:** $\text{ACME} / (\text{ACME} + \text{ADE})$
- 95% bootstrap confidence intervals for ACME ($n = 1{,}000$)

In [ ]:
# == Causal Mediation Analysis (Imai et al., 2010) =========================
G = df['gender_bin'].values  # 0=male, 1=female
Y = df['binary_label'].values

sc_med = StandardScaler()
M_all = sc_med.fit_transform(df[SHORTCUT_FEATURES].fillna(0).values)

def compute_mediation(G, Y, M):
    '''ACME, ADE, total via counterfactual mediation (binary outcome).'''
    lr_m = LinearRegression().fit(G.reshape(-1, 1), M)
    alpha = lr_m.coef_[0]

    X_ym = np.column_stack([G, M])
    lr_y = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_ym, Y)
    b0   = lr_y.intercept_[0]
    tau_p = lr_y.coef_[0, 0]
    gamma = lr_y.coef_[0, 1]

    lr_t = LogisticRegression(max_iter=1000, random_state=SEED).fit(
        G.reshape(-1, 1), Y)
    tau = lr_t.coef_[0, 0]

    m1 = lr_m.predict(np.ones((len(G), 1)))
    m0 = lr_m.predict(np.zeros((len(G), 1)))

    acme = (expit(b0 + tau_p*G + gamma*m1) -
            expit(b0 + tau_p*G + gamma*m0)).mean()
    ade  = (expit(b0 + tau_p*1 + gamma*M) -
            expit(b0 + tau_p*0 + gamma*M)).mean()
    return acme, ade, acme + ade, alpha, tau_p, gamma, tau

mediation_results = []
t0 = time.time()

for j, feat in enumerate(SHORTCUT_FEATURES):
    M = M_all[:, j]
    acme, ade, total, alpha, tau_p, gamma, tau = compute_mediation(G, Y, M)
    prop_med = acme / total if abs(total) > 1e-10 else 0.0

    rng = np.random.RandomState(SEED)
    n = len(G)
    boot_acmes = []
    for _ in range(N_BOOT):
        idx = rng.choice(n, n, replace=True)
        a_b, _, _, _, _, _, _ = compute_mediation(G[idx], Y[idx], M[idx])
        boot_acmes.append(a_b)

    ci_lo, ci_hi = np.percentile(boot_acmes, [2.5, 97.5])
    sig = '***' if (ci_lo > 0 or ci_hi < 0) else 'n.s.'

    mediation_results.append({
        'feature': feat,
        'alpha_G_to_M': alpha, 'gamma_M_to_Y': gamma,
        'tau_total': tau, 'tau_prime_direct': tau_p,
        'ACME': acme, 'ADE': ade,
        'total_effect': total, 'prop_mediated': prop_med,
        'ACME_CI_lo': ci_lo, 'ACME_CI_hi': ci_hi, 'sig': sig,
    })

    print(f'\n{feat}')
    print(f'  alpha (Gender->Mediator):  {alpha:+.4f}')
    print(f'  gamma (Mediator->Label):   {gamma:+.4f}')
    print(f'  ACME  (indirect):      {acme:+.6f}  95% CI [{ci_lo:+.6f}, {ci_hi:+.6f}] {sig}')
    print(f'  ADE   (direct):        {ade:+.6f}')
    print(f'  Total effect:          {total:+.6f}')
    print(f'  Proportion mediated:   {prop_med:.1%}')

med_df = pd.DataFrame(mediation_results)
print(f'\n{"="*60}')
print(f'Elapsed: {time.time()-t0:.1f}s')
print('ACME = indirect (G->M->Y) | ADE = direct (G->Y bypassing M)')
print('*** = 95% bootstrap CI excludes zero')


In [ ]:
# -- Mediation Visualization -----------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x_pos = np.arange(len(SHORTCUT_FEATURES))
w = 0.25
acmes  = med_df['ACME'].values
ades   = med_df['ADE'].values
totals = med_df['total_effect'].values
ci_lo  = med_df['ACME_CI_lo'].values
ci_hi  = med_df['ACME_CI_hi'].values

ax.bar(x_pos - w, acmes, w, label='ACME (indirect)', color='#e74c3c',
       alpha=0.85, edgecolor='black', lw=0.5)
ax.errorbar(x_pos - w, acmes, yerr=[acmes - ci_lo, ci_hi - acmes],
            fmt='none', color='black', capsize=4, lw=1.5)
ax.bar(x_pos, ades, w, label='ADE (direct)', color='#3498db',
       alpha=0.85, edgecolor='black', lw=0.5)
ax.bar(x_pos + w, totals, w, label='Total', color='#2c3e50',
       alpha=0.85, edgecolor='black', lw=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels([f.replace('_', '\n') for f in SHORTCUT_FEATURES], fontsize=10)
ax.set_ylabel('Effect Size (probability scale)')
ax.set_title('Causal Mediation Decomposition')
ax.legend(fontsize=9)
ax.axhline(0, color='black', lw=0.5)

ax = axes[1]
props = med_df['prop_mediated'].values * 100
colors_prop = ['#e74c3c' if s == '***' else '#bdc3c7' for s in med_df['sig']]
bars = ax.barh(range(len(SHORTCUT_FEATURES)), props, color=colors_prop,
               edgecolor='black', lw=0.5)
ax.set_yticks(range(len(SHORTCUT_FEATURES)))
ax.set_yticklabels(SHORTCUT_FEATURES, fontsize=11)
ax.set_xlabel('Proportion Mediated (%)')
ax.set_title('Fraction of Gender->Label via Each Shortcut')
for i, (bar, p) in enumerate(zip(bars, props)):
    sig = med_df.iloc[i]['sig']
    ax.text(max(bar.get_width() + 1, 2), bar.get_y() + bar.get_height()/2,
            f'{p:.1f}% {sig}', va='center', fontsize=10)
ax.set_xlim(0, max(max(props) * 1.5, 10))

plt.tight_layout()
plt.savefig('data/stage3/mediation_analysis.png', dpi=150, bbox_inches='tight')
print('Saved -> data/stage3/mediation_analysis.png')
plt.show()


## Part C — Concept Erasure (INLP) & Counterfactual Interventions

**INLP** (Iterative Nullspace Projection; Ravfogel et al., 2020) removes specific concept information from embeddings:

1. Train linear classifier to predict concept (e.g., `negative_emotion > median`) from [CLS]
2. Extract weight vector $\mathbf{w}$ (the concept direction)
3. Project embeddings onto null space: $\mathbf{X}' = \mathbf{X} - (\mathbf{X} \cdot \mathbf{w}) \mathbf{w}^\top$
4. Repeat until concept is unrecoverable (AUC < 0.52)

After erasure, we re-probe for:
- **The erased concept** → should drop dramatically (validation)
- **Gender** → if drops, the shortcut *mediates* gender encoding in [CLS]
- **Label** → if drops, the model *causally relies* on the shortcut for predictions

Then we apply the *same trained label classifier* to erased embeddings → **counterfactual prediction flips**, stratified by gender.

In [ ]:
# == INLP Concept Erasure + Post-Erasure Probing ===========================
def inlp_erase(X, y, max_iters=15, seed=42):
    '''Iterative Nullspace Projection: remove concept directions from X.'''
    X_res = X.copy()
    removed = 0
    for i in range(max_iters):
        rng = np.random.RandomState(seed + i)
        mask = rng.rand(len(y)) < 0.8
        clf = LogisticRegression(max_iter=300, C=1.0, solver='lbfgs',
                                 random_state=seed)
        clf.fit(X_res[mask], y[mask])
        try:
            auc = roc_auc_score(y[~mask], clf.predict_proba(X_res[~mask])[:, 1])
        except:
            break
        if auc < 0.52:
            print(f'    Converged at iter {i} (AUC={auc:.3f})')
            break
        clf.fit(X_res, y)
        w = clf.coef_[0].copy()
        w /= np.linalg.norm(w)
        X_res -= np.outer(X_res @ w, w)
        removed += 1
    return X_res, removed

def probe_auc(X, y, n_splits=5, seed=42):
    '''5-fold CV AUC for a logistic probe.'''
    clf = LogisticRegression(max_iter=500, C=1.0, random_state=seed)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return cross_val_score(clf, X, y, cv=cv, scoring='roc_auc').mean()

# -- Scale embeddings ------------------------------------------------------
scaler_cls = StandardScaler()
X_scaled = scaler_cls.fit_transform(X_cls_all)

y_gender = df['gender_bin'].values
y_label  = df['binary_label'].values

base_gender_auc = probe_auc(X_scaled, y_gender)
base_label_auc  = probe_auc(X_scaled, y_label)
print(f'Baseline probes on [CLS]:')
print(f'  Gender AUC = {base_gender_auc:.4f}')
print(f'  Label  AUC = {base_label_auc:.4f}\n')

# -- Per-shortcut erasure --------------------------------------------------
erased_embeddings = {}
erasure_results = []
t0 = time.time()

for feat in SHORTCUT_FEATURES:
    print('~' * 50)
    print(f'Erasing: {feat}')
    y_feat = (df[feat].fillna(0).values > df[feat].fillna(0).median()).astype(int)
    pre_feat_auc = probe_auc(X_scaled, y_feat)
    print(f'  Pre-erasure  {feat} AUC = {pre_feat_auc:.4f}')

    X_erased, n_removed = inlp_erase(X_scaled, y_feat)
    erased_embeddings[feat] = X_erased

    post_feat_auc   = probe_auc(X_erased, y_feat)
    post_gender_auc = probe_auc(X_erased, y_gender)
    post_label_auc  = probe_auc(X_erased, y_label)

    print(f'  Post-erasure {feat} AUC = {post_feat_auc:.4f}  (d={post_feat_auc-pre_feat_auc:+.4f})')
    print(f'  Post-erasure gender   AUC = {post_gender_auc:.4f}  (d={post_gender_auc-base_gender_auc:+.4f})')
    print(f'  Post-erasure label    AUC = {post_label_auc:.4f}  (d={post_label_auc-base_label_auc:+.4f})')
    print(f'  Directions removed: {n_removed}')

    erasure_results.append({
        'concept_erased': feat, 'n_dims_removed': n_removed,
        'pre_concept_AUC': pre_feat_auc, 'post_concept_AUC': post_feat_auc,
        'delta_concept': post_feat_auc - pre_feat_auc,
        'pre_gender_AUC': base_gender_auc, 'post_gender_AUC': post_gender_auc,
        'delta_gender': post_gender_auc - base_gender_auc,
        'pre_label_AUC': base_label_auc, 'post_label_AUC': post_label_auc,
        'delta_label': post_label_auc - base_label_auc,
    })

# -- Sequential erasure of ALL 7 shortcuts ---------------------------------
print('\n' + '~' * 50)
print('Sequential erasure of ALL 7 shortcuts:')
X_all_erased = X_scaled.copy()
total_removed = 0
for feat in SHORTCUT_FEATURES:
    y_f = (df[feat].fillna(0).values > df[feat].fillna(0).median()).astype(int)
    X_all_erased, n = inlp_erase(X_all_erased, y_f)
    total_removed += n
    print(f'  After erasing {feat}: +{n} directions (total={total_removed})')

erased_embeddings['ALL_shortcuts'] = X_all_erased
post_all_gender = probe_auc(X_all_erased, y_gender)
post_all_label  = probe_auc(X_all_erased, y_label)
print(f'\n  After ALL shortcut erasures:')
print(f'    Gender: {base_gender_auc:.4f} -> {post_all_gender:.4f}  (d={post_all_gender-base_gender_auc:+.4f})')
print(f'    Label:  {base_label_auc:.4f} -> {post_all_label:.4f}  (d={post_all_label-base_label_auc:+.4f})')

# -- Also erase GENDER itself (for total-effect comparison) ----------------
print('\n' + '~' * 50)
print('Erasing: gender (for total-effect comparison)')
X_gender_erased, n_g = inlp_erase(X_scaled, y_gender)
erased_embeddings['gender'] = X_gender_erased
post_g_gender = probe_auc(X_gender_erased, y_gender)
post_g_label  = probe_auc(X_gender_erased, y_label)
print(f'  Gender: {base_gender_auc:.4f} -> {post_g_gender:.4f}  (d={post_g_gender-base_gender_auc:+.4f})')
print(f'  Label:  {base_label_auc:.4f} -> {post_g_label:.4f}  (d={post_g_label-base_label_auc:+.4f})')

erasure_df = pd.DataFrame(erasure_results)
print(f'\nTotal elapsed: {time.time()-t0:.0f}s')
erasure_df


In [ ]:
# == Counterfactual Prediction Analysis + Visualization ====================
clf_orig = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)
clf_orig.fit(X_scaled, y_label)
probs_orig = clf_orig.predict_proba(X_scaled)[:, 1]
preds_orig = (probs_orig >= 0.5).astype(int)
print(f'Original model accuracy: {(preds_orig == y_label).mean():.3f}\n')

male_mask   = (y_gender == 0)
female_mask = (y_gender == 1)

erase_targets = SHORTCUT_FEATURES + ['ALL_shortcuts', 'gender']
flip_results = []

for target in erase_targets:
    X_er = erased_embeddings[target]
    probs_cf = clf_orig.predict_proba(X_er)[:, 1]
    preds_cf = (probs_cf >= 0.5).astype(int)
    flips = (preds_orig != preds_cf)
    prob_shift = probs_cf - probs_orig

    flip_results.append({
        'erased': target,
        'flip_rate': flips.mean(),
        'male_flip': flips[male_mask].mean(),
        'female_flip': flips[female_mask].mean(),
        'flip_asym': flips[female_mask].mean() - flips[male_mask].mean(),
        'mean_dp': prob_shift.mean(),
        'male_dp': prob_shift[male_mask].mean(),
        'female_dp': prob_shift[female_mask].mean(),
    })
    print(f'{target:20s}  flip={flips.mean():.2%}  m={flips[male_mask].mean():.2%}  '
          f'f={flips[female_mask].mean():.2%}  asym={flips[female_mask].mean()-flips[male_mask].mean():+.2%}')

flip_df = pd.DataFrame(flip_results)

# -- Visualization ---------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Panel 1: AUC drop after concept erasure
ax = axes[0]
n_feat = len(SHORTCUT_FEATURES)
x = np.arange(n_feat)
w = 0.22
drops_c = erasure_df['delta_concept'].values
drops_g = erasure_df['delta_gender'].values
drops_l = erasure_df['delta_label'].values

ax.bar(x - w, drops_c, w, label='Feature itself', color='#e74c3c',
       alpha=0.85, edgecolor='black', lw=0.5)
ax.bar(x,     drops_g, w, label='Gender',         color='#3498db',
       alpha=0.85, edgecolor='black', lw=0.5)
ax.bar(x + w, drops_l, w, label='Label',          color='#2ecc71',
       alpha=0.85, edgecolor='black', lw=0.5)
ax.set_xticks(x)
ax.set_xticklabels([f.replace('_', '\n') for f in SHORTCUT_FEATURES], fontsize=10)
ax.set_ylabel('Delta AUC After Erasure')
ax.set_title('Concept Erasure: Probe AUC Changes')
ax.legend(fontsize=8, loc='lower left')
ax.axhline(0, color='black', lw=0.5)

# Panel 2: Counterfactual flip rates by gender
ax = axes[1]
show = SHORTCUT_FEATURES + ['ALL_shortcuts']
show_df = flip_df[flip_df['erased'].isin(show)]
x2 = np.arange(len(show))
w2 = 0.3
ax.bar(x2 - w2/2, show_df['male_flip'].values * 100, w2,
       label='Male', color='#3498db', alpha=0.85, edgecolor='black', lw=0.5)
ax.bar(x2 + w2/2, show_df['female_flip'].values * 100, w2,
       label='Female', color='#e74c3c', alpha=0.85, edgecolor='black', lw=0.5)
ax.set_xticks(x2)
ax.set_xticklabels([s.replace('_', '\n') for s in show], fontsize=9)
ax.set_ylabel('Prediction Flip Rate (%)')
ax.set_title('Counterfactual Flips After Erasure')
ax.legend()

# Panel 3: Probability shift by gender
ax = axes[2]
ax.bar(x2 - w2/2, show_df['male_dp'].values, w2,
       label='Male', color='#3498db', alpha=0.85, edgecolor='black', lw=0.5)
ax.bar(x2 + w2/2, show_df['female_dp'].values, w2,
       label='Female', color='#e74c3c', alpha=0.85, edgecolor='black', lw=0.5)
ax.set_xticks(x2)
ax.set_xticklabels([s.replace('_', '\n') for s in show], fontsize=9)
ax.set_ylabel('Mean Probability Shift')
ax.set_title('Counterfactual P(positive) Shift')
ax.legend()
ax.axhline(0, color='black', lw=0.5)

plt.tight_layout()
plt.savefig('data/stage3/concept_erasure_analysis.png', dpi=150, bbox_inches='tight')
print('\nSaved -> data/stage3/concept_erasure_analysis.png')
plt.show()


## Part D — Conditional Independence Tests

Test the **d-separation** hypothesis from the SCM:

1. **Label $\perp$ Gender | Shortcuts?** — Does adding gender improve label prediction beyond what shortcuts already explain? (Likelihood ratio test + AUC comparison)
2. **Gender $\perp$ Label | Shortcuts?** — Does knowing the label help predict gender beyond what shortcuts explain?
3. **Using all 14 features** — Does gender add predictive power when all linguistic features are available?

If shortcuts *fully mediate* the gender→label path, adding gender should NOT improve prediction.

In [ ]:
# == Conditional Independence Tests ========================================
def log_lik(clf, X, y):
    '''Binary cross-entropy log-likelihood.'''
    p = np.clip(clf.predict_proba(X)[:, 1], 1e-10, 1 - 1e-10)
    return np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

X_s   = StandardScaler().fit_transform(df[SHORTCUT_FEATURES].fillna(0).values)
G_col = df['gender_bin'].values
Y_col = df['binary_label'].values
X_sg  = np.column_stack([X_s, G_col])

# -- Test 1: Label _|_ Gender | Shortcuts (7 features)? -------------------
print('Test 1: Label _|_ Gender | Shortcuts (7 features)?')
clf_r = LogisticRegression(max_iter=1000, C=100, random_state=SEED).fit(X_s, Y_col)
clf_f = LogisticRegression(max_iter=1000, C=100, random_state=SEED).fit(X_sg, Y_col)

ll_r, ll_f = log_lik(clf_r, X_s, Y_col), log_lik(clf_f, X_sg, Y_col)
lr_stat = 2 * (ll_f - ll_r)
p_val = 1 - chi2.cdf(lr_stat, df=1)

auc_r = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=SEED),
    X_s, Y_col, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring='roc_auc').mean()
auc_f = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=SEED),
    X_sg, Y_col, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring='roc_auc').mean()

print(f'  LR stat = {lr_stat:.2f},  p = {p_val:.2e}')
print(f'  AUC(Y~S) = {auc_r:.4f}  ->  AUC(Y~S+G) = {auc_f:.4f}  (d = {auc_f-auc_r:+.4f})')
verdict1 = 'REJECT independence (gender adds info)' if p_val < 0.05 \
    else 'Cannot reject independence'
print(f'  -> {verdict1}\n')

# -- Test 2: Gender _|_ Label | Shortcuts? ---------------------------------
print('Test 2: Gender _|_ Label | Shortcuts?')
X_sl = np.column_stack([X_s, Y_col])
clf_r2 = LogisticRegression(max_iter=1000, C=100, random_state=SEED).fit(X_s, G_col)
clf_f2 = LogisticRegression(max_iter=1000, C=100, random_state=SEED).fit(X_sl, G_col)
ll_r2, ll_f2 = log_lik(clf_r2, X_s, G_col), log_lik(clf_f2, X_sl, G_col)
lr_stat2 = 2 * (ll_f2 - ll_r2)
p_val2 = 1 - chi2.cdf(lr_stat2, df=1)

print(f'  LR stat = {lr_stat2:.2f},  p = {p_val2:.2e}')
verdict2 = 'REJECT (label adds info about gender)' if p_val2 < 0.05 \
    else 'Cannot reject (shortcuts explain gender-label assoc.)'
print(f'  -> {verdict2}\n')

# -- Test 3: Label _|_ Gender | ALL 14 features? --------------------------
print('Test 3: Label _|_ Gender | ALL 14 features?')
X_14  = StandardScaler().fit_transform(df[ALL_14_FEATURES].fillna(0).values)
X_14g = np.column_stack([X_14, G_col])

auc_14 = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=SEED),
    X_14, Y_col, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring='roc_auc').mean()
auc_14g = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=SEED),
    X_14g, Y_col, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    scoring='roc_auc').mean()

clf_r3 = LogisticRegression(max_iter=1000, C=100, random_state=SEED).fit(X_14, Y_col)
clf_f3 = LogisticRegression(max_iter=1000, C=100, random_state=SEED).fit(X_14g, Y_col)
lr_stat3 = 2 * (log_lik(clf_f3, X_14g, Y_col) - log_lik(clf_r3, X_14, Y_col))
p_val3 = 1 - chi2.cdf(lr_stat3, df=1)

print(f'  AUC(Y~14feat) = {auc_14:.4f}  ->  AUC(Y~14feat+G) = {auc_14g:.4f}  (d = {auc_14g-auc_14:+.4f})')
print(f'  LR stat = {lr_stat3:.2f},  p = {p_val3:.2e}')
verdict3 = 'REJECT independence' if p_val3 < 0.05 else 'Cannot reject independence'
print(f'  -> {verdict3}')

print(f'\n{"="*60}')
print('Interpretation:')
print('  If we REJECT independence: shortcuts do NOT fully mediate')
print('  the gender->label association. Gender carries additional info.')
print('  This means debiasing shortcuts alone may be insufficient.')


In [ ]:
# == STAGE 3 SUMMARY -- Save all results ==================================
os.makedirs('data/stage3', exist_ok=True)

med_df.to_csv('data/stage3/stage3_mediation_results.csv', index=False)
erasure_df.to_csv('data/stage3/stage3_concept_erasure_results.csv', index=False)
flip_df.to_csv('data/stage3/stage3_counterfactual_flips.csv', index=False)

print('=' * 60)
print('STAGE 3 SUMMARY -- CAUSAL DISENTANGLEMENT')
print('=' * 60)

print('\n(A) STRUCTURAL CAUSAL MODEL')
print('  DAG: Gender -> Shortcuts -> Embedding -> Prediction')
print('       Gender -> Clinical Signal -> Embedding -> Prediction')

print('\n(B) CAUSAL MEDIATION ANALYSIS')
for _, r in med_df.iterrows():
    print(f'  {r["feature"]:20s}  ACME={r["ACME"]:+.6f} '
          f'[{r["ACME_CI_lo"]:+.6f},{r["ACME_CI_hi"]:+.6f}]  '
          f'Prop.med={r["prop_mediated"]:.1%}  {r["sig"]}')

print('\n(C) CONCEPT ERASURE (INLP) -- AUC changes after erasure:')
for _, r in erasure_df.iterrows():
    print(f'  Erase {r["concept_erased"]:20s}  concept d={r["delta_concept"]:+.4f}  '
          f'gender d={r["delta_gender"]:+.4f}  label d={r["delta_label"]:+.4f}  '
          f'({r["n_dims_removed"]} dims)')

print('\n(D) COUNTERFACTUAL PREDICTION FLIPS:')
for _, r in flip_df.iterrows():
    print(f'  Erase {r["erased"]:20s}  flip={r["flip_rate"]:.1%}  '
          f'male={r["male_flip"]:.1%}  female={r["female_flip"]:.1%}  '
          f'asym={r["flip_asym"]:+.1%}')

print('\nSaved:')
for f in ['stage3_mediation_results.csv', 'stage3_concept_erasure_results.csv',
          'stage3_counterfactual_flips.csv', 'causal_dag.png',
          'mediation_analysis.png', 'concept_erasure_analysis.png']:
    print(f'  -> data/stage3/{f}')
